# Baseline-модели

В этом ноутбуке обучаются простые baseline-модели без feature engineering. Для обучения используются только исходные признаки после очистки данных: возраст, рост, вес, показатели артериального давления и категориальные признаки.

Baseline-модели нужны как начальная точка сравнения. Они позволяют получить первое значение качества и понять, какой результат дают простые подходы до усложнения модели и подбора гиперпараметров.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.modeling import (
    build_knn_baseline,
    build_logistic_regression_baseline,
    evaluate_classifier,
    save_experiment_table,
)
from src.preprocessing import (
    make_train_val_test_split,
    split_features_target,
)

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cardio_clean.csv"
EXPERIMENTS_PATH = PROJECT_ROOT / "report" / "tables" / "baseline_results.csv"

## Подготовка данных

В этом ноутбуке используется очищенный датасет, сохранённый после этапа EDA. Повторная очистка исходных данных здесь не выполняется, чтобы baseline-модели обучались на той же версии данных, которая была подготовлена и проверена ранее.

Далее данные разделяются на `train`, `validation` и `test` той же функцией и с тем же `random_state`, которые использовались при проверке split в EDA. Это позволяет воспроизвести выбранную схему разбиения.

Масштабирование числовых признаков, заполнение возможных пропусков и one-hot encoding категориальных признаков выполняются внутри `sklearn Pipeline`. Благодаря этому преобразования настраиваются только на `train` и не используют информацию из `validation` или `test`.

In [2]:
df_clean = pd.read_csv(PROCESSED_DATA_PATH)

x, y = split_features_target(df_clean)

x_train, x_val, x_test, y_train, y_val, y_test = make_train_val_test_split(x, y)

## Обучение baseline-моделей

Обучаются две простые модели: логистическая регрессия и KNN. Сравнение моделей выполняется только на `validation`, чтобы не использовать `test` на этапе выбора решения.

Основной метрикой для сравнения является `ROC-AUC`, так как она оценивает способность модели разделять классы по вероятностям и не зависит от выбранного порога классификации.

In [3]:
models = {
    "Logistic Regression": build_logistic_regression_baseline(),
    "KNN": build_knn_baseline(n_neighbors=15),
}

fitted_models = {}
validation_rows = []

for model_name, model in models.items():
    model.fit(x_train, y_train)
    fitted_models[model_name] = model

    metrics = evaluate_classifier(model, x_val, y_val)
    validation_rows.append(
        {
            "model": model_name,
            "split": "validation",
            **metrics,
        }
    )

validation_results = pd.DataFrame(validation_rows)
validation_results.sort_values("roc_auc", ascending=False)


,model,split,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,validation,0.726504,0.755730,0.660707,0.705031,0.793162
1,KNN,validation,0.718923,0.733333,0.678585,0.704898,0.779598


На validation-выборке обе baseline-модели показывают близкое качество: `F1-score` почти одинаковый, около 0.705.

По основной метрике `ROC-AUC` лучше работает логистическая регрессия: `0.793` против `0.780` у KNN. Также у логистической регрессии выше `accuracy` и `precision`, то есть она немного точнее отделяет положительный класс и делает меньше ложноположительных срабатываний.

KNN показывает немного более высокий `recall`, но разница небольшая. Поскольку основной метрикой выбрана `ROC-AUC`, в качестве лучшей baseline-модели выбирается `Logistic Regression`.

## Финальная проверка на test

После сравнения моделей на `validation` выбирается лучшая baseline-модель по основной метрике `ROC-AUC`. Затем выбранная модель один раз оценивается на `test`.

In [4]:
best_row = validation_results.sort_values("roc_auc", ascending=False).iloc[0]
best_model_name = best_row["model"]
best_model = fitted_models[best_model_name]

test_metrics = evaluate_classifier(best_model, x_test, y_test)

test_results = pd.DataFrame(
    [
        {
            "model": best_model_name,
            "split": "test",
            **test_metrics,
        }
    ]
)

baseline_results = save_experiment_table(
    pd.concat([validation_results, test_results], ignore_index=True).to_dict("records"),
    EXPERIMENTS_PATH,
)

split_order = {"validation": 0, "test": 1}

baseline_results_display = (
    baseline_results
    .assign(split_order=baseline_results["split"].map(split_order))
    .sort_values(["split_order", "roc_auc"], ascending=[True, False])
    .drop(columns="split_order")
    .reset_index(drop=True)
)

baseline_results_display


,model,split,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,validation,0.726504,0.755730,0.660707,0.705031,0.793162
1,KNN,validation,0.718923,0.733333,0.678585,0.704898,0.779598
2,Logistic Regression,test,0.726433,0.752777,0.665685,0.706557,0.790111


Лучшей baseline-моделью по `ROC-AUC` на validation-выборке стала `Logistic Regression`. После выбора модели она была один раз проверена на test-выборке.

На test-выборке логистическая регрессия показала качество, близкое к validation: `ROC-AUC` около 0.790 против 0.793 на validation, `F1-score` около 0.707 против 0.705. Это говорит о том, что качество модели стабильно и сильного расхождения между validation и test не наблюдается.

Полученный результат можно использовать как baseline для дальнейших экспериментов: новые модели или настройки должны сравниваться с этим уровнем качества.

## Вывод

В этом ноутбуке были обучены и сравнены baseline-модели без добавления новых признаков. Лучший результат на `validation` показала логистическая регрессия, поэтому именно она была выбрана для финальной проверки на `test`.

Качество на `test` оказалось близким к качеству на `validation`, значит baseline-модель работает достаточно стабильно. Полученные значения метрик будут использоваться как начальный ориентир: последующие модели и настройки должны улучшать качество относительно этого уровня, в первую очередь по основной метрике `ROC-AUC`.

Если более сложная модель улучшает результат только на `validation`, но не показывает сопоставимого улучшения на `test`, такой результат нельзя считать устойчивым.